# Día 3: Polígonos

**Reto:** Mapas coropléticos (Normalización).
**Datos:** AGEBs o Manzanas Urbanas de CDMX (Censo 2020).

El objetivo es unir los polígonos de las AGEBs con los indicadores sociodemográficos del censo y generar un mapa de intensidades (coroplético).

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Configuraciones visuales básicas
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Definir rutas a los archivos
path_shp = '../00_Datos/poligono_ageb_urbanas_cdmx/poligono_ageb_urbanas_cdmx.shp'
path_csv = '../00_Datos/ageb_mza_urbana_09_cpv2020/conjunto_de_datos/conjunto_de_datos_ageb_urbana_09_cpv2020.csv'

# 1. Cargar el shapefile (Polígonos de las AGEBs)
gdf_ageb = gpd.read_file(path_shp)

# 2. Cargar el CSV (Datos sociodemográficos del Censo)
# NOTA: los datos de INEGI suelen venir con códigos na ('*') que se pueden convertir en nulos.
# Como no estamos seguros todavía, los leemos con baja suposición por si hay advertencias de Dtype.
df_censo = pd.read_csv(path_csv, na_values=['*', 'N/A', 'N/D'], low_memory=False)

print(f"Shapefile: {gdf_ageb.shape}")
print(f"CSV Censo: {df_censo.shape}")

In [ ]:
# Vistazo al GeoDataFrame
gdf_ageb.head(3)

In [ ]:
# Vistazo al DataFrame del Censo
df_censo.head(3)

## 3. Preparación de los Datos para el Mapa

Para nuestro análisis hemos elegido crear el **Índice de Envejecimiento**, que nos dirá qué AGEBs tienen una población más longeva respecto a la población joven (0 a 14 años). Esto es útil para ubicar servicios, hospitales, farmacias, etc.

**Fórmula:** `(Población 60 e más / Población 0 a 14) * 100`

In [ ]:
# 1. Filtrar a nivel AGEB (MZA = 0)
df_ageb = df_censo[df_censo['MZA'] == 0].copy()

# 2. Crear la Llave Primaria (CVEGEO) de 13 dígitos
# Nos aseguramos que todos sean texto y se rellenen con ceros a la izquierda
df_ageb['CVE_ENT'] = df_ageb['ENTIDAD'].astype(str).str.zfill(2)
df_ageb['CVE_MUN'] = df_ageb['MUN'].astype(str).str.zfill(3)
df_ageb['CVE_LOC'] = df_ageb['LOC'].astype(str).str.zfill(4)
# AGEB ya viene como texto (es alfanumérico) en el Censo, debe tener 4 caracteres
df_ageb['CVE_AGEB_censo'] = df_ageb['AGEB'].astype(str).str.zfill(4)

# Unimos todo
df_ageb['CVEGEO'] = df_ageb['CVE_ENT'] + df_ageb['CVE_MUN'] + df_ageb['CVE_LOC'] + df_ageb['CVE_AGEB_censo']

print(f"AGEBs filtrados en Censo: {df_ageb.shape[0]}")
print(f"Ejemplo de CVEGEO Censo: {df_ageb['CVEGEO'].iloc[0]}")

## 4. Cálculo del Índice e Identificación de Valores Atípicos (EDA)

Aquí vamos a calcular matemáticamente el índice. 
**¡Ojo!** En el Análisis de Datos es fundamental revisar qué sucede cuando dividimos entre cero. Algunas AGEBs pueden tener población de adultos mayores pero cero niños.

In [ ]:
# 3. Calcular el Índice de Envejecimiento
import numpy as np

cols_jovenes = ['P_0A2', 'P_3A5', 'P_6A11', 'P_12A14']

# Convertimos a numérico (INEGI usa '*' para datos confidenciales, se vuelven NaN)
for col in cols_jovenes + ['P_60YMAS']:
    df_ageb[col] = pd.to_numeric(df_ageb[col], errors='coerce')

df_ageb['POB_JOVEN'] = df_ageb[cols_jovenes].sum(axis=1)

# Calculamos el índice
df_ageb['INDICE_ENVEJECIMIENTO'] = (df_ageb['P_60YMAS'] / df_ageb['POB_JOVEN']) * 100

# Análisis Exploratorio (EDA) rápido de los resultados:
print("---> ESTADÍSTICAS DEL ÍNDICE <--- ")
print(df_ageb['INDICE_ENVEJECIMIENTO'].describe())

# ¿Qué pasa con los infinitos?
num_inf = (df_ageb['INDICE_ENVEJECIMIENTO'] == np.inf).sum()
print(f"\n¡Alerta! Tenemos {num_inf} AGEBs con división por cero (0 jóvenes y >0 adultos mayores).")

### Limpieza y Distribución Visual
Si no corregimos esos valores infinitos, **el mapa de Matplotlib se romperá** (no sabe cómo pintar el infinito) y nos dejará un mapa de un solo color, como vimos antes.

Vamos a convertirlos a nulos (`NaN`) temporalmente para que el mapa y los histogramas los ignoren en su escala de color.

In [ ]:
# Reemplazamos los infinitos por NaN
df_ageb['INDICE_ENVEJECIMIENTO'] = df_ageb['INDICE_ENVEJECIMIENTO'].replace([np.inf, -np.inf], np.nan)

# Veamos cómo se distribuyen los datos de las AGEBs normales
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(df_ageb['INDICE_ENVEJECIMIENTO'].dropna(), bins=50, kde=True, color='purple')
plt.title('Distribución del Índice de Envejecimiento en las AGEBs de CDMX', fontsize=14)
plt.xlabel('Adultos Mayores por cada 100 Jóvenes')
plt.ylabel('Frecuencia (Cantidad de AGEBs)')
plt.axvline(100, color='red', linestyle='--', label='100 (La misma cantidad de jóvenes que adultos mayores)')
plt.legend()
plt.show()

print("Interpretación:")
print("> 100 significa que hay MÁS adultos mayores que niños.")
print("< 100 significa que hay MÁS niños que adultos mayores.")

In [ ]:
# 4. Hacer el Merge (Unir Mapa con Datos limpios)
mapa_ageb = gdf_ageb.merge(df_ageb[['CVEGEO', 'INDICE_ENVEJECIMIENTO']], on='CVEGEO', how='left')

print(f"Total de polígonos unidos: {mapa_ageb.shape[0]}")
print(f"Polígonos con datos válidos para pintar: {mapa_ageb['INDICE_ENVEJECIMIENTO'].notnull().sum()}")

In [ ]:
# 5. ¡Graficar el Mapa Coroplético Corregido!
fig, ax = plt.subplots(1, 1, figsize=(15, 12))

ax.set_title("Índice de Envejecimiento por AGEB\nen la Ciudad de México (Censo 2020)", fontsize=20, fontweight='bold', pad=20)
ax.axis('off')

# Pintar fondo base (Gris para los que tienen NaN o Infinito)
mapa_ageb[mapa_ageb['INDICE_ENVEJECIMIENTO'].isnull()].plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.2)

# Pintar el coroplético con los rangos definidos por los datos reales
mapa_ageb.dropna(subset=['INDICE_ENVEJECIMIENTO']).plot(
    column='INDICE_ENVEJECIMIENTO', 
    cmap='magma_r', 
    linewidth=0.1, 
    edgecolor='white', 
    legend=True,
    ax=ax,
    legend_kwds={'label': "Índice (Adultos Mayores por cada 100 Jóvenes)\n> 100 = Zonas envejecidas", 'orientation': "horizontal", 'shrink': 0.6}
)

plt.tight_layout()
plt.show()